# Act 1 — RDS, managed relational

Almost every application needs a relational database. Postgres, MySQL, SQL Server — the durable, transactional system of record where the authoritative state lives. You could run the engine yourself on EC2 — install the binary, patch the OS, set up replication, write backup scripts, build failover automation — and end up with significant operational surface. Or you could hand all of that to AWS and just bring the schema and the queries.

**RDS** is the second option. This act walks the six engines RDS supports, the storage layer underneath, the backup model and point-in-time restore, the two ways RDS gives you availability and read scaling (Multi-AZ and read replicas — which look similar and solve completely different problems), the security knobs, and **RDS Proxy** — the connection pool that makes Lambda-plus-relational actually work.

## The relational data tier on AWS

RDS is AWS's managed relational database service — you bring the schema and the queries; AWS handles the OS, the engine binary, patching, backups, and failover. Aurora is AWS's own cloud-native engine that speaks MySQL and PostgreSQL on the wire but has rebuilt the storage and replication layer from scratch. ElastiCache is the third piece — managed in-memory caching with Redis (or Valkey or Memcached) underneath — that sits in front of the database to absorb hot reads and turn millisecond queries into microsecond cache hits.

## RDS Overview

RDS runs the database engine on EC2 instances behind the scenes, with EBS for storage, and exposes a connection endpoint, a maintenance window, and a small set of configuration knobs. Everything else — patching the OS, upgrading the engine, replacing failed hardware, taking backups, setting up replicas, flipping Multi-AZ on failure — is AWS's job. You don't get SSH; there is no OS-level access. The cost of that lockdown is that any feature AWS hasn't certified (a database extension, a custom patch, Oracle RAC, a kernel-level monitoring agent) isn't reachable.

### Engines

Six supported engines:

- **MySQL**
- **PostgreSQL**
- **MariaDB**
- **Oracle** (BYOL or License Included)
- **Microsoft SQL Server** (License Included or BYOL via specific options)
- **IBM Db2**

All six get the same managed plane — automated backups, Multi-AZ, read replicas, Performance Insights, parameter groups, option groups — though the specific feature matrix per engine varies (e.g. read replicas aren't supported on every SQL Server edition).

### RDS vs self-managed on EC2

| | RDS | EC2 + DB engine |
|---|---|---|
| OS access | no SSH | full |
| Patching | AWS handles | you handle |
| Backups | automated | you build |
| Multi-AZ failover | checkbox | you build |
| Read replicas | built-in | you build |
| Engine version | AWS-supported list only | any version |
| Hourly cost | higher | lower |
| Total ops cost | usually lower | usually higher |

The right reason to pick EC2 over RDS is a feature RDS doesn't expose. For everything else, RDS wins once you include the cost of operations.

### Storage

RDS sits on EBS. `gp3` is the default general-purpose volume; `io1`/`io2` are for very-high-IOPS workloads (roughly above 64,000 IOPS); `gp2` is legacy. **Storage Autoscaling** grows the volume automatically when free space falls below a threshold, up to a ceiling you set — no downtime, no manual intervention. The right default for any workload whose growth isn't perfectly predictable.

## Backups and Point-in-Time Restore

RDS runs two backup mechanisms at the same time.

**Automated backups** are on by default. RDS takes a daily snapshot during the maintenance window and ships transaction logs to S3 every five minutes. Retention is **1 to 35 days** (0 disables them). Within that window you can do a **point-in-time restore (PITR)** to any second — the snapshot is the base and the logs replay forward. Automated backups are tied to the instance; when you delete the instance, they go with it unless you take a final snapshot.

**Manual snapshots** are triggered by you or by automation (Lambda, EventBridge, AWS Backup). They persist until you delete them — instance deletion doesn't touch them. Snapshots can be **copied to another region** (the standard cross-region DR pattern) and **shared with other AWS accounts**, optionally re-encrypted across the share.

The restore behaviour to remember: **restoring always creates a new instance**. PITR doesn't rewind the existing database in place; it spins up a parallel one at the target timestamp. The application then gets repointed at the new endpoint. Same for snapshot restores — never an in-place rollback.

**AWS Backup** is the cross-service backup orchestrator. It manages RDS snapshots alongside EBS, DynamoDB, EFS, FSx, and S3 backups — central policy, central retention, cross-region copies, cross-account vaults. The right answer when one compliance story has to span the estate.

| Backup type | Retention | Survives instance delete? | PITR? | Cross-region |
|---|---|---|---|---|
| Automated | 1–35 days | no (unless final snapshot) | yes | via copy |
| Manual snapshot | until you delete | yes | no (snapshot is a moment) | via copy |

## Multi-AZ vs Read Replicas

The two RDS features that look similar and solve completely different problems.

**Multi-AZ** provisions a **synchronous standby** in a different AZ in the same region. Every write commits to both primary and standby before the application sees an acknowledgement. The standby carries no traffic — it sits idle as a hot spare. When the primary fails (hardware, AZ outage, maintenance), RDS detects it, promotes the standby, and flips the DNS record on the DB endpoint. The application reconnects to the same endpoint name and lands on the new primary. Typical failover is **60 to 120 seconds**. The whole purpose is **high availability**, not capacity.

The newer **Multi-AZ DB Cluster** variant runs one writer plus two readable standbys across three AZs. The standbys are readable — so you get HA plus a little read scaling — and failover is faster (often under 35 seconds) thanks to having two candidates to promote.

**Read Replicas** are **asynchronous** copies that *do* serve read traffic. They have their own endpoint; the application explicitly routes reads to them. Up to **5 per RDS instance** (15 per Aurora cluster), in the same AZ, a different AZ, or **a different region**. Each replica has a small replication lag, so reads from it may return slightly stale data. Replicas can be **promoted** to standalone primaries — that breaks replication, useful for DR cutover or for splitting off a migration target.

| | Multi-AZ (classic) | Read Replica |
|---|---|---|
| Purpose | high availability | read scaling / DR |
| Replication | synchronous | asynchronous |
| Serves reads? | no | yes |
| Automatic failover? | yes | no — manual promotion |
| Endpoint | same — DNS flips | separate |
| Cross-region? | no | yes |
| Cost driver | second standby instance | per replica |

You combine them: Multi-AZ for HA in the primary region, read replicas for scaling read load, plus a cross-region read replica when a warm DR site in another region matters.

## RDS Security

**Network.** RDS instances live inside a VPC, normally in private subnets. Security groups gate inbound traffic — the canonical pattern allows the engine's port (5432, 3306, 1521, etc.) from the application tier's security group only. There is no production reason for an RDS instance to be publicly reachable.

**Encryption at rest** is KMS-based, AES-256. It must be enabled **at instance creation** — you cannot toggle it on later. To encrypt an existing unencrypted instance, the workflow is: take a snapshot, copy the snapshot with encryption enabled, restore the encrypted copy as a new instance, repoint the application. Read replicas of an encrypted primary are automatically encrypted with the same key.

**Encryption in transit** is enforced through the engine — SSL/TLS connection requirements set via the parameter group or the connection string. RDS publishes a regional CA bundle that the client validates against.

**IAM database authentication** lets the application skip the password entirely. It asks STS or the RDS API for a **short-lived auth token** (valid 15 minutes) and uses that token where a password would normally go. Supported on MySQL and PostgreSQL. Pairs naturally with EC2 instance profiles and Lambda execution roles — no DB credential in any config file, ever.

**Secrets Manager** is the standard place for master credentials when IAM DB auth isn't available (Oracle, SQL Server, legacy clients). RDS has a built-in integration that handles **rotation**: Secrets Manager periodically runs a rotation Lambda that updates the password on both RDS and the stored secret atomically. The application always reads the current secret rather than hardcoding it. SSM Parameter Store can hold credentials too, but Secrets Manager is the right tool when rotation matters.

## RDS Proxy

Relational databases have a hard cap on concurrent connections — opening one is expensive (TLS handshake, auth, memory allocation), and the server can only juggle so many before performance falls off a cliff. Long-running application servers handle this naturally with their own connection pool. **Lambda** doesn't: every invocation is its own process with its own short-lived connection. At a few thousand concurrent invocations the database starts rejecting connections, queries slow down, and the system collapses under its own bursting.

**RDS Proxy** is the fix. It's a managed connection pool that sits between the application and the database, multiplexing thousands of incoming application connections into a small pool of long-lived backend connections.

Three properties worth remembering.

- **Failover handling.** When Multi-AZ fails over, RDS Proxy detects it and reconnects to the new primary in seconds while the application's connection to the proxy stays open. Application code sees a brief pause rather than dropped sockets.
- **IAM auth + Secrets Manager.** The proxy pulls credentials from Secrets Manager and presents an IAM-authenticated endpoint to the application. Passwords never leave the proxy.
- **VPC-only.** The proxy endpoint is not publicly accessible — it has to live in your VPC.

RDS Proxy supports both RDS and Aurora, MySQL and PostgreSQL. The mental shortcut: **Lambda plus relational database equals put RDS Proxy in between.**

# Act 2 — Aurora, AWS's own engine

RDS runs the standard open-source engines on EC2 with EBS underneath — solid, but constrained by what those engines do on a single node with a single disk.

**Aurora** speaks the MySQL and PostgreSQL wire protocols on the outside, but the storage and replication layer is reimplemented from scratch. The result is a different performance ceiling, a different availability story, and a different cost model. The headline AWS publishes — five times the throughput of MySQL on RDS, three times the throughput of PostgreSQL — comes from one architectural decision: separating compute from storage and putting six copies of the data across three AZs underneath every cluster.

This act walks Aurora's storage architecture and what falls out of it — the endpoints, Serverless v2, Global Database for cross-region replication, Backtrack for in-place rewinds — and then the RDS-versus-Aurora decision.

## Aurora — a Different Engine

Aurora isn't a managed open-source MySQL or PostgreSQL. It's AWS's own engine that speaks the MySQL and PostgreSQL wire protocols on the front but reimplements the storage and replication layer underneath. That rebuild is where its performance and availability properties come from.

AWS's headline numbers: **5× the throughput of MySQL on RDS, 3× the throughput of PostgreSQL on RDS** at the same instance type. The mechanism is the **shared cluster volume**.

In a traditional database, compute and storage live together — each instance has its own data files on its own disk, and replication means streaming write events to other instances which then re-execute them. Aurora separates the two. The compute instances — primary and replicas — share a single **cluster volume**, a distributed log-structured storage layer that lives independently of the compute tier.

The cluster volume keeps **six copies of every piece of data across three AZs** — two copies per AZ. Writes acknowledge when **four of six** copies confirm; reads need only **three of six**. The storage layer is self-healing: corrupted blocks are continuously detected and repaired from the other copies. Storage **grows automatically in 10 GB increments up to 128 TB** — there's no concept of pre-provisioning Aurora storage. You're billed for what you actually use.

Two consequences fall out of this design.

**Adding a read replica copies no data.** A new replica is just a new compute node pointed at the same shared volume. Aurora supports up to **15 read replicas per cluster** (vs 5 on RDS), and replica lag is typically **under 100 ms** because replicas read from the same storage rather than waiting for redo-log shipping.

**Failover is fast.** In RDS Multi-AZ, failover means rebuilding the writer on the standby's volume — a minute or two. In Aurora, the replicas are already attached to the cluster volume; promoting one is a control-plane flip. **Aurora failover is typically under 30 seconds, often under 10.**

## Aurora Endpoints

An Aurora cluster exposes several endpoints — they look like normal DNS names, but each one routes differently.

| Endpoint | Routes to | Use for |
|---|---|---|
| **Cluster (writer)** | the current primary | writes |
| **Reader** | load-balanced across all replicas | reads |
| **Custom** | a subset of instances you define | e.g. heavy reports on a larger instance class |
| **Instance** | one specific instance | debugging, maintenance |

The application connects to the writer endpoint for writes and the reader endpoint for reads. If the primary fails and a replica is promoted, both endpoints update automatically — application code doesn't have to know which instance is currently the writer. The reader endpoint round-robins across the replicas, so it's a DNS-level load balancer for reads without you managing one.

**Custom endpoints** are the lever for mixed workloads on the same cluster — for example, two `r6g.4xlarge` replicas dedicated to overnight analytics, with the main reader endpoint pointing only at the smaller everyday replicas. Define a custom endpoint, attach the heavy replicas to it, and the BI tool connects there.

## Aurora Serverless v2

Aurora Serverless v2 auto-scales an Aurora cluster's compute capacity in real time, in fine-grained increments, based on actual load. The unit is the **Aurora Capacity Unit (ACU)** — roughly 2 GB of RAM plus proportional CPU and network — and the scaling step is **0.5 ACU**, applied in seconds without dropping connections.

You set a **min and max ACU range** for the cluster. During quiet periods it sits near the minimum; under load it scales up. The difference from v1 (the older, deprecated version) is significant: v1 paused and resumed by replacing the instance, with connection drops and a 10+ second warmup. v2 scales the *live* instance, so the application sees no break.

Fits naturally:

- variable or unpredictable workloads where provisioning for peak is wasteful
- dev / test / staging clusters
- multi-tenant SaaS where any one tenant is small but the aggregate bursts
- new applications where load is still unknown

You can mix Serverless v2 and provisioned instances in the same cluster — for example, a provisioned writer for steady write load with Serverless v2 readers that scale with read traffic.

## Aurora Global Database

Aurora Global Database spans multiple AWS regions with a single logical Aurora cluster. One **primary region** accepts writes; up to **five secondary regions** accept reads. Replication happens at the **storage layer**, not at the SQL level — typical lag is **under one second across continents**.

The DR numbers are the headline. **RPO** (data loss on disaster) is **under one second**; **RTO** (time to recover) is **under one minute** via managed planned failover. You can also do an unplanned **manual detach** — promote a secondary region to a standalone primary even if the original region is fully unreachable.

Each secondary region can have its own read replicas (up to 16 per region), giving regional users local-latency reads against a globally consistent dataset. The whole arrangement is the standard pattern for globally distributed applications and the strongest cross-region DR option Aurora offers.

## Backtrack and Multi-Master

Two Aurora features worth knowing but rarely needed.

**Aurora Backtrack** rewinds an Aurora MySQL cluster to a previous point in time **in place** — without restoring from a snapshot, without creating a new instance, without changing the cluster endpoint. The backtrack window is configurable up to **72 hours**. The whole operation takes seconds because Aurora's storage layer keeps a continuous change log. The use case: somebody just ran `DELETE FROM customers` without a `WHERE`. Backtrack five minutes, all good. **Aurora MySQL only — not PostgreSQL.**

**Aurora Multi-Master** lets multiple instances accept writes simultaneously, so a writer failure doesn't pause writes at all. The trade-off is that the application has to reason about conflicts at the row level. It's a specialised feature for very-high-write-availability workloads, available only in older Aurora MySQL versions. For almost every workload the standard single-writer-with-fast-failover model is the right answer.

## RDS vs Aurora — Picking Which

Both are managed relational databases on AWS. The choice comes down to engine, performance ceiling, and how variable the workload is.

| Pick | When |
|---|---|
| **RDS** (Oracle / SQL Server / Db2) | the engine isn't on Aurora's list |
| **RDS MySQL / PostgreSQL** | cost-sensitive, predictable workload, fine with 60–120 s failover |
| **Aurora MySQL / PostgreSQL** | want faster failover, faster replication, up to 15 replicas |
| **Aurora Serverless v2** | variable workload, multi-tenant SaaS, dev/test, unknown scale |
| **Aurora Global Database** | cross-region DR with RPO < 1 s, or global low-latency reads |
| **Aurora Backtrack** | guard against accidental data wipes on Aurora MySQL |

Cost shape matters too. Aurora is priced per-second compute plus per-GB-month storage plus per-million IO requests. RDS is per-second instance plus pre-provisioned EBS storage. For very small steady workloads RDS often comes out cheaper; for anything bursty or growing, Aurora's storage model usually wins because you pay for what you actually use rather than what you provisioned.

In [ ]:
import boto3

rds = boto3.client("rds", region_name="us-east-1")

# Multi-AZ Postgres on RDS — synchronous standby, encrypted, IAM auth enabled, 7-day PITR
rds.create_db_instance(
    DBInstanceIdentifier="prod-postgres",
    DBInstanceClass="db.t3.medium",
    Engine="postgres",
    EngineVersion="16.1",
    MasterUsername="admin",
    MasterUserPassword="use-secrets-manager-instead",
    AllocatedStorage=100,
    StorageType="gp3",
    StorageEncrypted=True,
    MultiAZ=True,
    DBSubnetGroupName="my-db-subnet-group",
    VpcSecurityGroupIds=["sg-0dbaccess"],
    BackupRetentionPeriod=7,
    EnableIAMDatabaseAuthentication=True,
    DeletionProtection=True,
)

# Same-region read replica — async; serves reads from its own endpoint
rds.create_db_instance_read_replica(
    DBInstanceIdentifier="prod-postgres-replica",
    SourceDBInstanceIdentifier="prod-postgres",
    DBInstanceClass="db.t3.medium",
    AvailabilityZone="us-east-1b",
)

# Aurora PostgreSQL cluster with Serverless v2 writer — ACUs scale 0.5 to 16
rds.create_db_cluster(
    DBClusterIdentifier="prod-aurora",
    Engine="aurora-postgresql",
    EngineVersion="16.1",
    MasterUsername="admin",
    MasterUserPassword="use-secrets-manager-instead",
    DatabaseName="appdb",
    DBSubnetGroupName="my-db-subnet-group",
    VpcSecurityGroupIds=["sg-0dbaccess"],
    StorageEncrypted=True,
    BackupRetentionPeriod=7,
    EnableIAMDatabaseAuthentication=True,
    ServerlessV2ScalingConfiguration={"MinCapacity": 0.5, "MaxCapacity": 16.0},
    DeletionProtection=True,
)
rds.create_db_instance(
    DBInstanceIdentifier="prod-aurora-writer",
    DBClusterIdentifier="prod-aurora",
    DBInstanceClass="db.serverless",
    Engine="aurora-postgresql",
)

# RDS Proxy — connection pool in front of the cluster, IAM-authed, TLS-required
rds.create_db_proxy(
    DBProxyName="aurora-proxy",
    EngineFamily="POSTGRESQL",
    Auth=[{
        "AuthScheme": "SECRETS",
        "SecretArn": "arn:aws:secretsmanager:us-east-1:111122223333:secret:aurora-creds",
        "IAMAuth": "REQUIRED",
    }],
    RoleArn="arn:aws:iam::111122223333:role/rds-proxy-role",
    VpcSubnetIds=["subnet-0priv1a", "subnet-0priv1b"],
    VpcSecurityGroupIds=["sg-0proxy"],
    RequireTLS=True,
)

# PITR — restores to a NEW instance at any second within the retention window
rds.restore_db_instance_to_point_in_time(
    SourceDBInstanceIdentifier="prod-postgres",
    TargetDBInstanceIdentifier="prod-postgres-restored",
    RestoreTime="2026-04-15T10:30:00Z",
    DBInstanceClass="db.t3.medium",
)

# Act 3 — ElastiCache, microseconds in front of the database

A database query takes a millisecond at best, hundreds of milliseconds under load. An in-memory cache lookup takes **microseconds** — three or four orders of magnitude faster. For any read pattern where the same answer is being computed over and over, putting a cache in front of the database is the single highest-leverage performance change available on AWS.

**ElastiCache** is the managed in-memory cache. It runs Redis, the newer Valkey fork, or Memcached underneath. This act covers why caching matters, the standard caching patterns (lazy loading, write-through, TTL, session store), the Redis-versus-Memcached choice, and how Redis itself is shaped — replication groups, cluster mode, the data structures that make Redis show up in so many architectures, and how to lock a production cluster down.

## Why Cache?

A database query takes a millisecond at best, hundreds of milliseconds under load. An in-memory cache lookup takes **microseconds** — three or four orders of magnitude faster. For any read pattern where the same answer is being computed over and over, putting a cache in front of the database is the single highest-leverage performance change available.

The shape: the application checks the cache first; on a **hit** it returns the cached value immediately. On a **miss** it falls through to the database, then writes the result into the cache before returning. Subsequent requests within the cache's TTL get the fast path.

AWS offers two services for this. **ElastiCache** is managed Redis, Valkey, or Memcached — general-purpose, works in front of anything. **DynamoDB Accelerator (DAX)** is a write-through cache purpose-built for DynamoDB and only DynamoDB — drop-in for the DynamoDB client, no application changes. The rest of this notebook is about ElastiCache.

## Caching Strategies

A handful of patterns cover almost every real cache usage.

**Lazy loading (cache-aside).** The application owns the logic. Read from cache; on miss, read from DB and write back to cache; return. Writes go straight to the DB and either invalidate or update the cache entry. The advantages: the cache only holds what's actually being read, and a cache outage doesn't break the application — it falls through to the slower DB path. The disadvantages: the first request for any key is always a miss (cold), and the cache can hold stale data until the entry expires or is explicitly invalidated.

**Write-through.** Every write goes to both the cache and the database. The cache stays consistent with the source of truth; reads never miss for data that's been written. The cost is a double-write penalty on every write, and you may end up caching data that's never read.

**Time-to-live (TTL).** Every cache entry carries an expiry. After the TTL the entry is automatically evicted and the next read goes back to the source. TTL is the cheap lossy answer to consistency — short enough that stale data isn't a real problem, long enough that the hit rate stays high. Usually combined with lazy loading.

**Session store.** Store session state in the cache rather than in any one application server's memory. Any server can read any session, so the application tier is **stateless** and auto-scales freely. Sessions fit naturally on a key-value model with a TTL matching the session timeout.

**Eviction policies.** When the cache fills up, something has to give. Redis defaults to LRU (least-recently-used), which matches most workloads. LFU (least-frequently-used), random, and TTL-based eviction are also available. Pick LRU unless the workload has a specific reason to prefer something else.

## Redis vs Memcached

ElastiCache offers two engines. They look superficially similar (both key-value, both in-memory) and choosing wrong is one of the more common architecture mistakes.

| Feature | Redis | Memcached |
|---|---|---|
| Data structures | strings, hashes, lists, sets, **sorted sets**, bitmaps, HyperLogLog, streams, geo | strings only |
| Persistence | RDB snapshots + AOF logs | no |
| Replication | yes — Multi-AZ with automatic failover | no |
| Sharding (cluster mode) | yes — up to 500 shards | yes — horizontal scaling |
| Pub/Sub, Lua, transactions | yes | no |
| Backup / restore | yes | no |
| Threading | mostly single-threaded | multi-threaded |
| Use when... | anything beyond basic K/V | pure simple cache, multi-thread throughput matters |

**The default answer is Redis.** Persistence, replication, Multi-AZ failover, sorted-set leaderboards, pub/sub, Lua scripts, transactions — all on Redis. Memcached is the right answer only when the workload is genuinely simple key-value caching that you're happy to lose entirely on a restart, and the multi-threaded throughput of a single Memcached node fits better than the single-threaded model of a single Redis node.

The newer **Valkey** engine — a Redis fork that ElastiCache now offers — is API-compatible with Redis and the right pick on cost: Valkey clusters are noticeably cheaper than the equivalently sized Redis cluster. Everything that's true of Redis below applies to Valkey too.

## ElastiCache for Redis

### Replication groups

A **replication group** is the unit of deployment for Redis on ElastiCache — one **primary** node plus up to five **read replicas**, optionally spread across AZs. With **Multi-AZ** enabled, ElastiCache automatically promotes a replica to primary on failure. The application connects to the **primary endpoint** for writes and reads, and optionally to the **reader endpoint** to load-balance reads across replicas.

### Cluster mode

**Cluster Mode Disabled** is the simple model: one shard (one primary plus replicas), all data on one logical key space. Up to about 500 GB. Right for any workload that fits on a single primary.

**Cluster Mode Enabled** shards the keyspace across up to **500 shards**, each with its own primary and replicas. Keys are distributed by hash slot. The application connects to the **cluster configuration endpoint** and the client library handles routing. Cluster mode scales write throughput linearly across shards and lets the dataset grow well past single-node memory limits — TB-plus ranges. The trade-off is that multi-key operations (transactions, Lua scripts) become harder because keys may live on different shards.

### Data structures worth knowing

Redis is more than a key-value store — its data structures are the reason it shows up in so many architectures.

| Use case | Data structure |
|---|---|
| Session / object cache | string (JSON serialised) or hash |
| Leaderboard / ranking | **sorted set** — O(log N) rank lookup |
| Rate limiter | string with `INCR` + `EXPIRE` |
| Recent activity feed | list (`LPUSH` + `LTRIM`) |
| Approximate unique counts | HyperLogLog |
| Pub/Sub messaging | channels |
| Geospatial queries | geo commands |

The **sorted set** is the headline. It stores members with a numeric score and keeps them ranked automatically: `ZADD` adds or updates a score; `ZREVRANK` returns the rank in O(log N); `ZREVRANGE` returns the top N. Real-time game leaderboards across millions of players come down to a couple of one-line calls.

### Persistence and security

Redis can persist to disk via **RDB snapshots** (periodic point-in-time files, fast restore, some data loss between snapshots) or **AOF logs** (every write appended to a log, near-zero data loss, larger files, slower replay) — or no persistence at all for a pure cache. ElastiCache exposes both as configurable backup settings.

Security wraps the engine:

- **In-transit encryption** (TLS) on client connections
- **At-rest encryption** (KMS) on persisted snapshots
- **AUTH token** — a shared secret the client presents before issuing commands
- **IAM authentication** on Redis 7 and later — IAM-signed tokens instead of a static password

A locked-down cluster has all four turned on plus a tight security group on port 6379.

## ElastiCache for Memcached

Memcached is simpler than Redis on purpose. No replication, no persistence, no Multi-AZ failover — losing a node means losing whatever was on it, and the application rebuilds the data on the next cache miss.

What it offers in return: **multi-threading** that scales with the instance's CPU cores, and **horizontal scaling** by adding more nodes — keys are distributed across nodes via consistent hashing on the client side. The client uses **Auto Discovery** to learn which nodes are in the cluster, so adding or removing a node doesn't require an application redeploy.

The honest use case is small: a pure object cache where data doesn't matter on restart, the workload is hot enough that multi-threading per node really helps, and you want the simplest possible operational model. In modern AWS architectures, Redis (or Valkey) covers almost every case people reach for Memcached for.

In [ ]:
import boto3

ec = boto3.client("elasticache", region_name="us-east-1")

# Redis replication group: 1 primary + 2 replicas, Multi-AZ failover, TLS + AUTH + at-rest encryption
ec.create_replication_group(
    ReplicationGroupId="prod-redis",
    Description="Production Redis cache",
    Engine="redis",
    EngineVersion="7.1",
    CacheNodeType="cache.r7g.large",
    NumCacheClusters=3,
    AutomaticFailoverEnabled=True,
    MultiAZEnabled=True,
    CacheSubnetGroupName="my-cache-subnet-group",
    SecurityGroupIds=["sg-0cache"],
    AtRestEncryptionEnabled=True,
    TransitEncryptionEnabled=True,
    AuthToken="a-strong-token-at-least-32-chars-long",
    SnapshotRetentionLimit=7,
)

# Memcached cluster — pure cache, no replication, no persistence
ec.create_cache_cluster(
    CacheClusterId="dev-memcached",
    Engine="memcached",
    CacheNodeType="cache.t3.medium",
    NumCacheNodes=3,                       # client-side consistent hashing distributes keys
    CacheSubnetGroupName="my-cache-subnet-group",
    SecurityGroupIds=["sg-0cache"],
)

In [ ]:
# Common Redis patterns with redis-py — pip install redis
import json, time, redis

r = redis.Redis(
    host="prod-redis.abc123.ng.0001.use1.cache.amazonaws.com",
    port=6379,
    ssl=True,
    password="a-strong-token-at-least-32-chars-long",
    decode_responses=True,
)

# Lazy loading: cache on miss with a 5-minute TTL
def get_user(user_id: str) -> dict:
    key = f"user:{user_id}"
    cached = r.get(key)
    if cached:
        return json.loads(cached)
    user = fetch_from_db(user_id)            # <-- the slow path
    r.setex(key, 300, json.dumps(user))
    return user

# Rate limiter: INCR + EXPIRE on a windowed key
def is_rate_limited(user_id: str, limit: int = 100, window_s: int = 60) -> bool:
    key = f"ratelimit:{user_id}:{int(time.time()) // window_s}"
    count = r.incr(key)
    if count == 1:
        r.expire(key, window_s)              # set expiry only on first INCR
    return count > limit

# Sorted-set leaderboard: ZADD scores, then top-N or rank lookups
lb = "game:leaderboard"
r.zadd(lb, {"alice": 9500, "bob": 8200, "carol": 9800})
top3 = r.zrevrange(lb, 0, 2, withscores=True)
alice_rank = r.zrevrank(lb, "alice")          # 0-indexed from the top
r.zincrby(lb, 200, "alice")                   # atomic increment

# Act 4 — Putting the tier together

You now have three services and a set of choices within each. The question is how to stack them in a real architecture — what role each plays, where they overlap, and where they emphatically do not replace each other.

## Putting the Tier Together

A short framing for stacking RDS, Aurora, and ElastiCache in a single architecture.

**Source of truth.** Always relational on RDS or Aurora — durable, transactional, the system of record. Pick the engine first: if it's not on Aurora's list (Oracle, SQL Server, Db2), use RDS. If it's MySQL or PostgreSQL, Aurora is the default unless cost on a small steady workload nudges back to RDS.

**Availability.** Multi-AZ on RDS; multi-AZ replicas on Aurora. Cross-region requires read replicas (RDS) or Global Database (Aurora) — Aurora's Global Database is the strongest cross-region option for a relational store on AWS.

**Connection pooling.** RDS Proxy whenever Lambda talks to either RDS or Aurora. It's the standard part of any serverless-plus-relational stack.

**Read scaling.** Read replicas first (RDS up to 5, Aurora up to 15); if read replicas alone don't get you there, put **ElastiCache** in front of the read path. Cache the heavy repeated reads, leave the database to do real work.

**Microsecond latency.** Anything that needs sub-millisecond response — session lookups, rate-limit counters, hot leaderboards — belongs in ElastiCache, not in the database. The database is for durable state; the cache is for the hot subset.

These three services don't replace each other. They stack.